# 05 Python Unit Tests - pytest - parametryzacja i markery
_Kamil Bartocha_ | _wersja 2.0_

## Rozkład jazdy

1. 🔹 **`@pytest.mark.parametrize`** - wiele przypadkow testowych
2. 🔹 **Markery wbudowane** - `skip`, `xfail`, `skipif`
3. 🔹 **Własne markery i filtrowanie testow** - `-m`, `pytest.ini`

🐍 Każda sekcja zawiera ćwiczenia.

## 1. 🔹 `@pytest.mark.parametrize` - wiele przypadkow testowych

`@pytest.mark.parametrize` pozwala uruchamiac ten sam test
dla wielu zestawow danych - bez powtarzania kodu:

```python
@pytest.mark.parametrize('n, expected', [
    (0,  True),
    (2,  True),
    (1,  False),
    (-3, False),
])
def test_is_even(n, expected):
    assert is_even(n) == expected
```

Kazda krotka generuje osobny test widoczny w raporcie:
```
test_is_even[0-True]   PASSED
test_is_even[2-True]   PASSED
test_is_even[1-False]  PASSED
```

**Parametr `ids`** - wlasne nazwy zamiast domyslnych:
```python
@pytest.mark.parametrize('text', ['', '  ', None],
                          ids=['empty', 'spaces', 'none'])
```

**Zagniezdzone `@parametrize`** - iloczyn kartezjanski:
```python
@pytest.mark.parametrize('a', [1, 2])
@pytest.mark.parametrize('b', [10, 20])
def test_add(a, b):          # generuje 4 testy: (1,10),(1,20),(2,10),(2,20)
    assert add(a, b) == a + b
```

**Testowanie wyjatkow z parametrize:**
```python
@pytest.mark.parametrize('bad_input', [-1, 'abc', None])
def test_parse_raises(bad_input):
    with pytest.raises((ValueError, TypeError)):
        parse(bad_input)
```

> 💡 **Tip:** `@parametrize` zastepuje `subTest` z `unittest`
> i jest bardziej czytelny. Kazdy przypadek jest oddzielnym
> testem w raporcie - latwo zidentyfikowac ktory zawiodl.

In [ ]:
import subprocess
import sys
import tempfile
import os


def _run(code, flags=None):
    """Zapisz kod jako plik testowy i uruchom przez pytest."""
    flags = flags or ['-v', '--tb=short', '-p', 'no:cacheprovider']
    with tempfile.NamedTemporaryFile(
        mode='w', suffix='_test.py', delete=False, dir='.'
    ) as tmp:
        tmp.write(code)
        fname = tmp.name
    result = subprocess.run(
        [sys.executable, '-m', 'pytest', fname] + flags,
        capture_output=True, text=True
    )
    short = os.path.basename(fname)
    print(result.stdout.replace(fname, short)[-1500:])
    os.unlink(fname)


_run('''
import pytest


def is_even(n):
    return n % 2 == 0


def is_palindrome(text):
    return text == text[::-1]


# Jeden parametr - lista wartosci
@pytest.mark.parametrize('n', [0, 2, 4, 6, 8])
def test_is_even_for_even_numbers(n):
    assert is_even(n) is True


# Dwa parametry - lista krotek
@pytest.mark.parametrize('n, expected', [
    (0,  True),
    (2,  True),
    (1,  False),
    (-3, False),
    (7,  False),
])
def test_is_even(n, expected):
    assert is_even(n) == expected


# Wlasne IDs dla lepszych nazw testow
@pytest.mark.parametrize('text, expected', [
    ('racecar', True),
    ('hello',   False),
    ('',        True),
    ('level',   True),
], ids=['palindrome', 'not-palindrome', 'empty', 'level'])
def test_is_palindrome(text, expected):
    assert is_palindrome(text) == expected
''')

---

### 🐍 Ćwiczenia - `@pytest.mark.parametrize`

**Ćw. 1.** Przetestuj `is_even(n)` dla co najmniej 8 wartosci
uzywajac `@parametrize` - zarowno liczby parzyste jak i nieparzyste.

**Ćw. 2.** Parametryzuj test wyjatku dla `parse_int(text)` -
rozne bledne wejscia powinny rzucac `ValueError` lub `TypeError`.

**Ćw. 8.** Zagniezdz dwa `@parametrize` aby wygenerowac
iloczyn kartezjanski przypadkow dla operacji arytmetycznych.

In [ ]:
# Ćw. 1: is_even dla 8 wartosci przez @parametrize
_code = '''
import pytest


def is_even(n):
    return n % 2 == 0


# Uzupelnij - co najmniej 8 wartosci (parzyste i nieparzyste):
@pytest.mark.parametrize('n, expected', [
    (0,   True),
    (2,   ...),   # parzysta
    (4,   ...),   # parzysta
    (-2,  ...),   # ujemna parzysta
    (1,   False),
    (3,   ...),   # nieparzysta
    (-1,  ...),   # ujemna nieparzysta
    (100, ...),   # duza liczba
])
def test_is_even(n, expected):
    assert is_even(n) == expected
'''
_run(_code)

In [ ]:
# Ćw. 2: parametryzuj test wyjatku dla parse_int()
_code = '''
import pytest


def parse_int(text):
    """Parsuje napis na int. Rzuca ValueError lub TypeError."""
    if text is None:
        raise TypeError('Input cannot be None')
    if not isinstance(text, str):
        raise TypeError(f'Expected str, got {type(text).__name__}')
    text = text.strip()
    if not text:
        raise ValueError('Input cannot be empty')
    return int(text)   # rzuca ValueError dla nienumerycznych napisow


# Uzupelnij - rozne bledne wejscia:
@pytest.mark.parametrize('bad_input, exc_type', [
    ('abc',  ValueError),
    ('',     ...),   # hint: ValueError - pusty napis
    ('1.5',  ...),   # hint: ValueError - float jako napis
    (None,   ...),   # hint: TypeError
    (42,     ...),   # hint: TypeError - int zamiast str
    ('  ',   ...),   # hint: ValueError - same spacje
])
def test_parse_int_raises(bad_input, exc_type):
    # hint: with pytest.raises(exc_type):
    ...


# Uzupelnij - poprawne wejscia:
@pytest.mark.parametrize('text, expected', [
    ('42',   42),
    ('-5',   ...),
    (' 10 ', ...),   # hint: strip() usuwa spacje
])
def test_parse_int_valid(text, expected):
    assert parse_int(text) == expected
'''
_run(_code)

In [ ]:
# Ćw. 8: zagniezdzone @parametrize - iloczyn kartezjanski
_code = '''
import pytest


def clamp(value, lo, hi):
    """Ogranicza value do zakresu [lo, hi]."""
    return max(lo, min(value, hi))


# Zagniezdzone @parametrize generuje iloczyn kartezjanski
# hint: dwa @parametrize na tej samej funkcji = n*m testow
@pytest.mark.parametrize('lo, hi', [
    (0, 10),
    (-5, 5),
    ...   # hint: dodaj trzeci zakres
])
@pytest.mark.parametrize('value, expected_range', [
    (3,   True),   # hint: 3 powinno byc w kazdym zakresie
    (-10, False),  # hint: -10 poza pierwszym zakresem
    (100, False),  # hint: 100 powyzej wszystkich zakresow
])
def test_clamp_stays_in_range(value, expected_range, lo, hi):
    result = clamp(value, lo, hi)
    # hint: result powinien zawsze byc w [lo, hi]
    assert lo <= result <= hi


# Uzupelnij: zagniezdzone parametrize dla operacji arytmetycznych
@pytest.mark.parametrize('op, fn', [
    ('add', lambda a, b: a + b),
    ('sub', lambda a, b: a - b),
    ('mul', lambda a, b: a * b),
])
@pytest.mark.parametrize('a, b', [
    (2, 3),
    (0, 5),
    (-1, 4),
])
def test_arithmetic_result_is_numeric(a, b, op, fn):
    result = fn(a, b)
    assert isinstance(result, (int, float))
'''
_run(_code)

## 2. 🔹 Markery wbudowane: `skip`, `xfail`, `skipif`

Markery sa metadanymi przypisywanymi do testow. Wbudowane
markery pytest zmieniaja zachowanie testu podczas uruchomienia:

**`@pytest.mark.skip(reason=...)`** - zawsze pomija test:
```python
@pytest.mark.skip(reason='UI not implemented yet')
def test_render_dashboard():
    ...
```

**`@pytest.mark.skipif(condition, reason=...)`** - pomija
warunkowo:
```python
@pytest.mark.skipif(sys.version_info < (3, 10),
                    reason='Requires Python 3.10+')
def test_match_statement():
    ...
```

**`@pytest.mark.xfail`** - test oczekiwanie nie przechodzi
(dokumentuje znany blad):
```python
@pytest.mark.xfail(reason='Bug #42: divide rounds incorrectly')
def test_divide_precision():
    assert divide(1, 3) == 0.3333
```

**`@pytest.mark.xfail(strict=True)`** - test MUSI nie przejsc.
Jesli przejdzie - blad `XPASS` (unexpected success):
```python
@pytest.mark.xfail(strict=True, reason='Not implemented')
def test_future_feature():
    assert new_feature() == 'done'   # musi rzucic wyjatek
```

| Wynik | Symbol | Znaczenie |
|-------|--------|-----------|
| test pominiety | `s` | skip/skipif |
| spodziewana porazka | `x` | xfail - OK |
| nieoczekiwany sukces | `X` | xpass - blad gdy strict=True |

**`pytest.param(..., marks=...)`** - marker dla jednego
przypadku w parametrize:
```python
@pytest.mark.parametrize('n', [
    1,
    pytest.param(-1, marks=pytest.mark.xfail),
    pytest.param(None, marks=pytest.mark.xfail(raises=TypeError)),
])
def test_sqrt(n):
    assert sqrt(n) >= 0
```

> 💡 **Tip:** Uzyj `@xfail` do dokumentowania znanych bledow
> zamiast komentowania testu. Gdy blad zostanie naprawiony,
> test zacznie przechodzic - dostaniesz `XPASS` jako przypomnienie
> zeby usunac dekorator.

In [ ]:
_run('''
import sys
import pytest


def sqrt(n):
    if n < 0:
        raise ValueError(f'Cannot take sqrt of negative: {n}')
    return n ** 0.5


def parse_version(ver_str):
    # BUG: nie obsluguje sufiksow takich jak '3.10.1b2'
    return tuple(int(x) for x in ver_str.split('.'))


# --- skip ---
@pytest.mark.skip(reason='Funkcja sqrt_complex nie istnieje jeszcze')
def test_sqrt_complex_numbers():
    assert sqrt_complex(-4) == 2j


# --- skipif ---
@pytest.mark.skipif(sys.version_info < (3, 10),
                    reason='Wymaga Python >= 3.10')
def test_sqrt_on_current_python():
    assert sqrt(4) == pytest.approx(2.0)


# --- xfail - znany blad ---
@pytest.mark.xfail(reason='Bug: parse_version nie radzi sobie z sufiksami')
def test_parse_version_with_suffix():
    result = parse_version('3.10.1b2')
    assert result == (3, 10, 1)


# --- xfail strict - musi nie przejsc ---
@pytest.mark.xfail(strict=True, reason='Nie zaimplementowano')
def test_not_yet_implemented():
    raise NotImplementedError('todo')


# --- pytest.param z marks ---
@pytest.mark.parametrize('n', [
    4,
    9,
    pytest.param(-1, marks=pytest.mark.xfail(raises=ValueError)),
    pytest.param(-4, marks=pytest.mark.xfail(raises=ValueError)),
])
def test_sqrt_parametrized(n):
    result = sqrt(n)
    assert result >= 0
''')

---

### 🐍 Ćwiczenia - markery wbudowane

**Ćw. 3. *(Trudniejsze)*** Parametryzuj test z uzyciem
`pytest.param(..., marks=pytest.mark.xfail)` dla przypadkow,
ktore maja nie przechodzic (np. edge cases z bledami).

**Ćw. 4.** Pomiń test przez `@pytest.mark.skip(reason=...)`.
Dodaj rowniez `@pytest.mark.skipif` uzalezniajacy od wersji
Pythona lub systemu operacyjnego.

**Ćw. 5.** Oznacz test jako `@pytest.mark.xfail(strict=True)`
dla funkcji z celowym bledem. Sprawdz wynik. Nastepnie napraw
funkcje i sprawdz ze dostajemy `XPASS`.

In [ ]:
# Ćw. 3: pytest.param z marks=xfail w parametrize
_code = '''
import pytest


def safe_divide(a, b):
    """Dzieli a przez b. Rzuca ZeroDivisionError dla b=0."""
    if b == 0:
        raise ZeroDivisionError('division by zero')
    return a / b


# Uzupelnij: niektore przypadki maja nie przechodzic (xfail)
@pytest.mark.parametrize('a, b, expected', [
    (10, 2,  5.0),
    (9,  3,  3.0),
    (7,  2,  3.5),
    # hint: te przypadki maja rzucac wyjatek - oznacz je xfail
    pytest.param(1, 0, None,
                 marks=pytest.mark.xfail(raises=ZeroDivisionError)),
    pytest.param(10, 0, ...,
                 marks=...),   # hint: pytest.mark.xfail(raises=...)
    pytest.param(0,  0, ...,
                 marks=...),
])
def test_safe_divide(a, b, expected):
    result = safe_divide(a, b)
    assert result == pytest.approx(expected)


# Uzupelnij: test z mieszanymi markami (jeden xfail przez TypeError)
def process(value):
    if not isinstance(value, (int, float)):
        raise TypeError(f'Expected number, got {type(value).__name__}')
    return value * 2


@pytest.mark.parametrize('value, expected', [
    (5,     10),
    (2.5,   5.0),
    pytest.param('hello', None,
                 marks=...),   # hint: xfail(raises=TypeError)
    pytest.param(None, None,
                 marks=...),
])
def test_process(value, expected):
    assert process(value) == expected
'''
_run(_code)

In [ ]:
# Ćw. 4: @pytest.mark.skip i @pytest.mark.skipif
_code = '''
import sys
import pytest


def current_platform():
    return sys.platform


def parse_uuid(text):
    """Parsuje UUID. Nie zaimplementowane jeszcze."""
    raise NotImplementedError('TODO: implement UUID parsing')


def feature_flag(name):
    """Zwraca stan feature flagi."""
    flags = {'dark_mode': True, 'beta_ui': False}
    return flags.get(name, False)


# Uzupelnij: pomiń test z powodu WIP
@pytest.mark.skip(reason=...)   # hint: podaj przyczyne
def test_parse_uuid():
    result = parse_uuid('123e4567-e89b-12d3-a456-426614174000')
    assert len(result) == 32


# Uzupelnij: pomiń warunkowo dla Pythona < 3.10
@pytest.mark.skipif(..., reason='Wymaga Python >= 3.10')
def test_python_310_feature():
    # hint: warunek: sys.version_info < (3, 10)
    assert sys.version_info >= (3, 10)


# Uzupelnij: pomiń na Windowsie
@pytest.mark.skipif(..., reason='Tylko na Linux/Mac')
def test_unix_platform():
    # hint: warunek: sys.platform == 'win32'
    assert current_platform() in ('linux', 'darwin')


# Ten test zawsze sie uruchamia:
def test_feature_flag_dark_mode():
    assert feature_flag('dark_mode') is True
'''
_run(_code)

In [ ]:
# Ćw. 5: @pytest.mark.xfail(strict=True) - musi nie przejsc
_code = '''
import pytest


# Funkcja z celowym bledem - zwraca bledna wartosc
def buggy_median(numbers):
    """BUG: zwraca srednia zamiast mediany."""
    return sum(numbers) / len(numbers)   # powinno byc sorted()[len//2]


# Uzupelnij: oznacz test jako xfail - wiemy ze funkcja jest zepsuta
@pytest.mark.xfail(strict=..., reason='Bug: median zwraca srednia')
def test_buggy_median_odd_list():
    # hint: strict=True oznacza ze test MUSI nie przejsc
    result = buggy_median([1, 2, 10])
    assert result == 2   # mediana to 2, funkcja zwroci 4.33...


@pytest.mark.xfail(reason='Bug: nie obsluguje parzystej dlugosci')
def test_buggy_median_even_list():
    result = buggy_median([1, 2, 3, 4])
    assert result == 2.5   # mediana to 2.5, funkcja zwroci 2.5 (akurat!)
    # hint: ten test PRZEJDZIE (xpass) - srednia dla [1,2,3,4] = 2.5


# Po "naprawieniu" bledu (odkomentuj ponizej):
# def fixed_median(numbers):
#     s = sorted(numbers)
#     n = len(s)
#     return s[n // 2] if n % 2 else (s[n//2 - 1] + s[n//2]) / 2
#
# def test_fixed_median():
#     assert fixed_median([1, 2, 10]) == 2
'''
_run(_code)

## 3. 🔹 Własne markery i filtrowanie testow

**Wlasne markery** pozwalaja grupowac testy i uruchamiac je
selektywnie. Rejestracja w `pytest.ini` eliminuje ostrzezenia:

```ini
[pytest]
markers =
    slow: testy trwajace dluzej niz 1 sekunde
    db: testy wymagajace bazy danych
    smoke: podstawowe testy sanity check
```

Uzycie w kodzie:
```python
@pytest.mark.slow
def test_heavy_computation():
    ...

@pytest.mark.db
@pytest.mark.slow
def test_database_migration():
    ...
```

**Filtrowanie przez `-m`:**
```bash
pytest -m slow            # tylko slow
pytest -m 'not slow'      # wszystkie oprocz slow
pytest -m 'slow and db'   # slow ORAZ db
pytest -m 'slow or smoke' # slow LUB smoke
```

**`pytest-timeout`** - ograniczenie czasu wykonania testu:
```python
# pip install pytest-timeout

@pytest.mark.timeout(5)   # maksymalnie 5 sekund
def test_api_response():
    ...
```

Lub globalnie w `pytest.ini`:
```ini
[pytest]
timeout = 30
```

**Parametryzowana fixture** vs `@parametrize` - rownowazne
podejscia do testowania wielu wariantow:

| Sposob | Kiedy uzywac |
|--------|--------------|
| `@parametrize` | dane testowe (wejscie/wyjscie) |
| fixture z `params` | konfiguracja srodowiska/infrastruktury |

> 💡 **Tip:** Uzyj konwencji nazewniczej markerow pasujacych
> do struktury projektu: `unit`, `integration`, `e2e`, `slow`,
> `db`. Uruchamiaj `pytest -m 'not slow and not db'` w szybkim
> cyklu TDD i `pytest -m all` przed commitem.

In [ ]:
import os
import shutil
import subprocess
import sys
import tempfile


def _run_with_ini(test_code, ini_content, flags=None):
    """Uruchom testy z plikiem pytest.ini w katalogu tymczasowym."""
    flags = flags or ['-v', '--tb=short', '-p', 'no:cacheprovider']
    tmpdir = tempfile.mkdtemp()
    with open(os.path.join(tmpdir, 'pytest.ini'), 'w') as f:
        f.write(ini_content)
    with open(os.path.join(tmpdir, 'test_sample.py'), 'w') as f:
        f.write(test_code)
    result = subprocess.run(
        [sys.executable, '-m', 'pytest', tmpdir] + flags,
        capture_output=True, text=True
    )
    print(result.stdout[-1500:])
    shutil.rmtree(tmpdir)


_ini = '''
[pytest]
markers =
    slow: testy trwajace dluzej niz 1 sekunde
    smoke: podstawowe testy sanity check
    db: testy wymagajace bazy danych
'''

_test = '''
import pytest
import time


@pytest.mark.smoke
def test_addition():
    assert 1 + 1 == 2


@pytest.mark.smoke
def test_string_upper():
    assert 'hello'.upper() == 'HELLO'


@pytest.mark.slow
def test_heavy_operation():
    time.sleep(0.05)
    assert True


@pytest.mark.slow
@pytest.mark.db
def test_db_query():
    time.sleep(0.02)
    assert True
'''

print('=== pytest -m smoke ===')
_run_with_ini(_test, _ini, flags=['-v', '-m', 'smoke', '-p', 'no:cacheprovider'])

print('=== pytest -m "not slow" ===')
_run_with_ini(_test, _ini, flags=['-v', '-m', 'not slow', '-p', 'no:cacheprovider'])

print('=== pytest -m "slow and db" ===')
_run_with_ini(_test, _ini, flags=['-v', '-m', 'slow and db', '-p', 'no:cacheprovider'])

---

### 🐍 Ćwiczenia - własne markery i filtrowanie

**Ćw. 6. *(Trudniejsze)*** Zdefiniuj wlasny marker
`@pytest.mark.slow` i uruchom testy z `-m 'not slow'`.
Dodaj `pytest.ini` z zarejestrowanym markerem.

**Ćw. 7.** Napisz parametryzowana fixture `app_config` dla
dwoch srodowisk: `test` i `staging`. Ten sam test powinien
przejsc dla obu konfiguracji.

**Ćw. 9. *(Trudniejsze)*** Parametryzuj testy dla klasy
`Matrix` - operacje `add`, `transpose` i walidacja
dla roznych kształtow macierzy.

In [ ]:
# Ćw. 6: wlasny marker @pytest.mark.slow + filtrowanie
import os
import shutil
import subprocess
import sys
import tempfile


# Uzupelnij: pytest.ini z zarejestrowanym markerem slow
_ini = '''
[pytest]
markers =
    slow: ...
    smoke: ...
'''
# hint: dodaj opis dla kazdego markera

# Uzupelnij: plik testowy z markerami
_test = '''
import pytest
import time


# Uzupelnij: oznacz odpowiednie testy
def test_quick_add():
    assert 1 + 1 == 2


def test_quick_string():
    assert 'hello'.upper() == 'HELLO'


def test_slow_sort():
    import random
    data = list(range(1000))
    random.shuffle(data)
    assert sorted(data) == list(range(1000))


def test_slow_string_ops():
    result = ' '.join(str(i) for i in range(1000))
    assert len(result) > 0


def test_smoke_health_check():
    assert True
'''

tmpdir = tempfile.mkdtemp()
with open(os.path.join(tmpdir, 'pytest.ini'), 'w') as f:
    f.write(_ini)
with open(os.path.join(tmpdir, 'test_markers.py'), 'w') as f:
    f.write(_test)

# Uruchom wszystkie testy
print('=== wszystkie testy ===')
result = subprocess.run(
    [sys.executable, '-m', 'pytest', tmpdir, '-v',
     '-p', 'no:cacheprovider'],
    capture_output=True, text=True
)
print(result.stdout[-900:])

# Uzupelnij: uruchom tylko -m 'not slow'
print('\n=== -m not slow ===')
result = subprocess.run(
    [sys.executable, '-m', 'pytest', tmpdir,
     '-v', '-m', ...,  # hint: 'not slow'
     '-p', 'no:cacheprovider'],
    capture_output=True, text=True
)
print(result.stdout[-600:])
shutil.rmtree(tmpdir)

In [ ]:
# Ćw. 7: parametryzowana fixture dla dwoch srodowisk
_code = '''
import pytest


class AppConfig:
    def __init__(self, env, db_url, debug, timeout):
        self.env     = env
        self.db_url  = db_url
        self.debug   = debug
        self.timeout = timeout

    def is_valid(self):
        return bool(self.db_url) and self.timeout > 0


# Uzupelnij: fixture z params dla dwoch srodowisk
@pytest.fixture(params=[...])   # hint: ['test', 'staging']
def app_config(request):
    configs = {
        'test': AppConfig(
            env='test', db_url=':memory:',
            debug=True, timeout=5
        ),
        'staging': AppConfig(
            env='staging', db_url='postgresql://staging/app',
            debug=False, timeout=30
        ),
    }
    ...  # hint: return configs[request.param]


# Te testy uruchamiaja sie dla kazdego srodowiska:
def test_config_is_valid(app_config):
    assert app_config.is_valid()


def test_config_has_db_url(app_config):
    assert app_config.db_url is not None
    assert len(app_config.db_url) > 0


def test_config_timeout_positive(app_config):
    assert app_config.timeout > 0


def test_staging_is_not_debug(app_config):
    # hint: tylko staging powinno miec debug=False
    if app_config.env == 'staging':
        assert app_config.debug is False
'''
_run(_code)

In [ ]:
# Ćw. 9: parametryzuj testy dla klasy Matrix
_code = '''
import pytest


class Matrix:
    def __init__(self, data):
        self.data = data
        self.rows = len(data)
        self.cols = len(data[0]) if data else 0

    def add(self, other):
        if self.rows != other.rows or self.cols != other.cols:
            raise ValueError('Matrix dimensions must match for add')
        return Matrix([
            [self.data[r][c] + other.data[r][c]
             for c in range(self.cols)]
            for r in range(self.rows)
        ])

    def transpose(self):
        return Matrix([
            [self.data[r][c] for r in range(self.rows)]
            for c in range(self.cols)
        ])

    def __eq__(self, other):
        return self.data == other.data


# Uzupelnij: parametrize dla operacji add
@pytest.mark.parametrize('a_data, b_data, expected_data', [
    ([[1, 2], [3, 4]],
     [[5, 6], [7, 8]],
     [[6, 8], [10, 12]]),
    ([[0, 0], [0, 0]],
     [[1, 2], [3, 4]],
     ...),   # hint: dodaj, neutralny element dodawania
    ([[1]], [[2]], ...),  # hint: macierz 1x1
])
def test_matrix_add(a_data, b_data, expected_data):
    result = Matrix(a_data).add(Matrix(b_data))
    assert result == Matrix(expected_data)


# Uzupelnij: parametrize dla transpose
@pytest.mark.parametrize('original, transposed', [
    ([[1, 2, 3],
      [4, 5, 6]],
     [[1, 4],
      [2, 5],
      [3, 6]]),
    ([[1, 2], [3, 4]], ...),  # hint: transponuj macierz 2x2
    ([[5]],            ...),  # hint: macierz 1x1
])
def test_matrix_transpose(original, transposed):
    result = Matrix(original).transpose()
    assert result == Matrix(transposed)


# Uzupelnij: xfail dla niedopasowanych wymiarow
@pytest.mark.parametrize('a_data, b_data', [
    ([[1, 2]], [[1, 2, 3]]),   # 1x2 + 1x3
    pytest.param(
        [[1, 2, 3]], [[1], [2]],
        marks=pytest.mark.xfail(raises=ValueError)
    ),
    ...   # hint: dodaj wlasny przypadek xfail
])
def test_matrix_add_dimension_mismatch_raises(a_data, b_data):
    with pytest.raises(ValueError):
        Matrix(a_data).add(Matrix(b_data))
'''
_run(_code)